In [2]:
from statsmodels.stats.proportion import proportions_ztest
from scipy.stats import norm
import numpy as np

np.random.seed(42)


In [3]:
#Calculate how many users you need BEFORE starting
baseline_rate = 0.04
target_rate = 0.055
alpha = 0.05
power = 0.80

def cohens_h(p1, p2):
    return 2 * np.arcsin(np.sqrt(p1)) - 2 * np.arcsin(np.sqrt(p2))

effect_size = cohens_h(target_rate, baseline_rate)

z_alpha = norm.ppf(1 - alpha/2)
z_beta = norm.ppf(power)
n = ((z_alpha + z_beta) / effect_size) ** 2

print(f'Minimum sample size per group: {int(np.ceil(n))} users')
print(f'Total experiment size: {int(np.ceil(n)) * 2} users')
print(f'At 1,000 daily visitors (50/50 split), run for at least {int(np.ceil(n*2/500))} days')

Minimum sample size per group: 1569 users
Total experiment size: 3138 users
At 1,000 daily visitors (50/50 split), run for at least 7 days


In [4]:

#Simulate and analyze the A/B test
n_per_group = int(np.ceil(n))
control_conversions = np.random.binomial(1, baseline_rate, size=n_per_group)
treatment_conversions = np.random.binomial(1, target_rate, size=n_per_group)

conv_rate_A = control_conversions.mean()
conv_rate_B = treatment_conversions.mean()
print(f'Control (A): {conv_rate_A:.4f} ({conv_rate_A*100:.2f}%)')
print(f'Treatment (B): {conv_rate_B:.4f} ({conv_rate_B*100:.2f}%)')
print(f'Relative lift: {(conv_rate_B - conv_rate_A)/conv_rate_A * 100:.1f}%')

successes = [treatment_conversions.sum(), control_conversions.sum()]
nobs = [n_per_group, n_per_group]
z_stat, p_val = proportions_ztest(successes, nobs, alternative='larger')
print(f'\nz-statistic: {z_stat:.4f}')
print(f'p-value: {p_val:.4f}')
print(f'Decision: {"Reject H0 — roll out new design" if p_val < alpha else "Fail to reject H0"}')

diff = conv_rate_B - conv_rate_A
se = np.sqrt((conv_rate_A*(1-conv_rate_A) + conv_rate_B*(1-conv_rate_B)) / n_per_group)
ci_low = diff - 1.96 * se
ci_high = diff + 1.96 * se
print(f'95% CI for difference: [{ci_low:.4f}, {ci_high:.4f}]')
print('If CI does not include 0: statistically significant')

Control (A): 0.0427 (4.27%)
Treatment (B): 0.0625 (6.25%)
Relative lift: 46.3%

z-statistic: 2.4794
p-value: 0.0066
Decision: Reject H0 — roll out new design
95% CI for difference: [0.0042, 0.0354]
If CI does not include 0: statistically significant


In [6]:
#Why stopping early ruins your experiment
import warnings
warnings.filterwarnings("ignore", category=RuntimeWarning)

peeking_rejections = 0
n_simulations = 1000

for _ in range(n_simulations):
    a_data = np.random.binomial(1, baseline_rate, n_per_group)
    b_data = np.random.binomial(1, baseline_rate, n_per_group)
    for checkpoint in range(50, n_per_group + 1, 50):
        _, p = proportions_ztest(
            [b_data[:checkpoint].sum(), a_data[:checkpoint].sum()],
            [checkpoint, checkpoint]
        )
        if p < 0.05:
            peeking_rejections += 1
            break

print(f'False positive rate WITH peeking: {peeking_rejections/n_simulations:.3f}')
print(f'Expected false positive rate WITHOUT peeking: 0.050')
print('Peeking can more than double your false positive rate!')

False positive rate WITH peeking: 0.269
Expected false positive rate WITHOUT peeking: 0.050
Peeking can more than double your false positive rate!
